In [5]:
import pandas as pd

# Load dataset
url = "https://raw.githubusercontent.com/4GeeksAcademy/NLP-project-tutorial/main/url_spam.csv"

df = pd.read_csv(url)

print(df.head())
print(df.info())
print(df["is_spam"].value_counts())


                                                 url  is_spam
0  https://briefingday.us8.list-manage.com/unsubs...     True
1                             https://www.hvper.com/     True
2                 https://briefingday.com/m/v4n3i4f3     True
3   https://briefingday.com/n/20200618/m#commentform    False
4                        https://briefingday.com/fan     True
<class 'pandas.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   url      2999 non-null   str  
 1   is_spam  2999 non-null   bool 
dtypes: bool(1), str(1)
memory usage: 26.5 KB
None
is_spam
False    2303
True      696
Name: count, dtype: int64


Download NLTK resources

In [1]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...


True

URL Cleaning Function

In [6]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_url(text):

    # Lowercase
    text = text.lower()

    # Replace URL punctuation with spaces
    text = re.sub(r"[/:.?=&_-]", " ", text)

    # Keep letters and numbers
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)

    tokens = text.split()

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return " ".join(tokens)

df["clean_url"] = df["url"].apply(clean_url)

print(df[["url","clean_url"]].head())

                                                 url  \
0  https://briefingday.us8.list-manage.com/unsubs...   
1                             https://www.hvper.com/   
2                 https://briefingday.com/m/v4n3i4f3   
3   https://briefingday.com/n/20200618/m#commentform   
4                        https://briefingday.com/fan   

                                          clean_url  
0  http briefingday us8 list manage com unsubscribe  
1                                http www hvper com  
2                     http briefingday com v4n3i4f3  
3      http briefingday com n 20200618 mcommentform  
4                          http briefingday com fan  


Convert Text to Numbers

Machine learning models cannot understand text directly.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["clean_url"]
y = df["is_spam"]

vectorizer = TfidfVectorizer(max_features=5000)

X_vectorized = vectorizer.fit_transform(X)

Train/Test Split

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Step 3: Build the SVM Model
Basic SVM

In [9]:
from sklearn.svm import SVC

model = SVC()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

Evaluate

In [10]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print(classification_report(y_test, predictions))

Accuracy: 0.96
              precision    recall  f1-score   support

       False       0.96      0.99      0.97       461
        True       0.98      0.85      0.91       139

    accuracy                           0.96       600
   macro avg       0.97      0.92      0.94       600
weighted avg       0.96      0.96      0.96       600



Step 4: Optimize the Model

Use Grid Search.

In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

grid_search = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}


Best Model

In [12]:
best_model = grid_search.best_estimator_

predictions = best_model.predict(X_test)

print("Optimized Accuracy:")
print(accuracy_score(y_test, predictions))

print(classification_report(y_test, predictions))

Optimized Accuracy:
0.9716666666666667
              precision    recall  f1-score   support

       False       0.97      0.99      0.98       461
        True       0.98      0.90      0.94       139

    accuracy                           0.97       600
   macro avg       0.97      0.95      0.96       600
weighted avg       0.97      0.97      0.97       600



Step 5: Save the Model

In [ ]:
import os

os.makedirs("models", exist_ok=True)



In [18]:
import joblib

joblib.dump(best_model, "models/svm_model.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")

print("Model saved successfully!")

Model saved successfully!
